In [2]:
import pandas as pd

url_books = 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv'
url_ratings = 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv'

# TODO: books және ratings деректерін жүктеңіз
books = pd.read_csv(url_books)
ratings = pd.read_csv(url_ratings)

# Тексеру
print(f"Books: {books.shape}")
print(f"Ratings: {ratings.shape}")
books.head(3)

Books: (10000, 23)
Ratings: (5976479, 3)


,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...


In [3]:
# TODO: бірегей авторлар санын есептеңіз
unique_authors = books['authors'].nunique()
print(f"Бірегей авторлар саны: {unique_authors}")

# TODO: орташа рейтингті есептеңіз
avg_rating = ratings['rating'].mean()
print(f"Орташа рейтинг: {avg_rating:.2f}")

# TODO: бос мәндер санын шығарыңыз
print("\nБос мәндер:")
print(books.isnull().sum())

Бірегей авторлар саны: 4664
Орташа рейтинг: 3.92

Бос мәндер:
book_id                         0
goodreads_book_id               0
best_book_id                    0
work_id                         0
books_count                     0
isbn                          700
isbn13                        585
authors                         0
original_publication_year      21
original_title                585
title                           0
language_code                1084
average_rating                  0
ratings_count                   0
work_ratings_count              0
work_text_reviews_count         0
ratings_1                       0
ratings_2                       0
ratings_3                       0
ratings_4                       0
ratings_5                       0
image_url                       0
small_image_url                 0
dtype: int64


In [4]:
# Бос мәндерді толтыру
books['authors'] = books['authors'].fillna('')
books['title'] = books['title'].fillna('')

# TODO: tags бағанын құрыңыз
books['tags'] = (books['title'] + " " + books['authors']).str.lower()

# Тексеру
print("Алғашқы 5 тег:")
for i in range(5):
    print(f"  [{i}] {books.loc[i, 'tags'][:80]}")

Алғашқы 5 тег:
  [0] the hunger games (the hunger games, #1) suzanne collins
  [1] harry potter and the sorcerer's stone (harry potter, #1) j.k. rowling, mary gran
  [2] twilight (twilight, #1) stephenie meyer
  [3] to kill a mockingbird harper lee
  [4] the great gatsby f. scott fitzgerald


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TfidfVectorizer құрып, tfidf_matrix есептеңіз
tfidf = TfidfVectorizer(max_features=3000)
tfidf_matrix = tfidf.fit_transform(books['tags'])

# Тексеру
print(f"TF-IDF матрица өлшемі: {tfidf_matrix.shape}")

TF-IDF матрица өлшемі: (10000, 3000)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similar_books(book_index, tfidf_matrix, top_n=5):
    # TODO: cosine_similarity арқылы ұқсастықты есептеңіз
    sim_scores = cosine_similarity(tfidf_matrix[book_index:book_index+1], tfidf_matrix).flatten()

    # TODO: ең ұқсас top_n индексті табыңыз (өзін-өзі алып тастап)
    top_indices = sim_scores.argsort()[::-1][1:top_n+1]

    return top_indices, sim_scores[top_indices]

# Тексеру
idx = 0
top_idx, top_sim = get_similar_books(idx, tfidf_matrix, top_n=5)
print(f"'{books.loc[idx, 'title']}' кітабына ұқсас кітаптар:")
for i, (bi, si) in enumerate(zip(top_idx, top_sim), 1):
    print(f"  {i}. {books.loc[bi, 'title'][:50]:50s} | sim={si:.4f}")

'The Hunger Games (The Hunger Games, #1)' кітабына ұқсас кітаптар:
  1. The Hunger Games Trilogy Boxset (The Hunger Games, | sim=0.9767
  2. Mockingjay (The Hunger Games, #3)                  | sim=0.9502
  3. Catching Fire (The Hunger Games, #2)               | sim=0.8816
  4. The World of the Hunger Games (Hunger Games Trilog | sim=0.7727
  5. The Hunger Games Tribute Guide                     | sim=0.6826


In [12]:
def recommend(title_query, books, tfidf_matrix, top_n=5):
    # TODO: books кестесінде title_query жолын іздеңіз
    matches = books[books['title'].str.contains(title_query, case=False, na=False)]

    if matches.empty:
        print(f"'{title_query}' табылмады.")
        return

    idx = matches.index[0]
    print(f"Кітап: '{books.loc[idx, 'title']}' -- {books.loc[idx, 'authors']}")
    print(f"Орташа рейтинг: {books.loc[idx, 'average_rating']}\n")

    top_idx, top_sim = get_similar_books(idx, tfidf_matrix, top_n)

    print("Ұсынылған кітаптар:")
    for i, (bi, si) in enumerate(zip(top_idx, top_sim), 1):
        b = books.loc[bi]
        print(f"  {i}. {b['title'][:45]:45s} | {b['authors'][:25]:25s} | sim={si:.3f}")

In [13]:
# TODO: 3 кітап үшін recommend() функциясын шақырыңыз
recommend('Hunger Games', books, tfidf_matrix)
recommend('Hobbit', books, tfidf_matrix)
recommend('The Great Gatsby', books, tfidf_matrix)

# TODO: Бір сөйлем жазыңыз
# Жауап: Ұсыныстар өте мағыналы, себебі жүйе негізінен сол автордың басқа кітаптарын немесе ұқсас жанрдағы туындыларды тауып берді.

Кітап: 'The Hunger Games (The Hunger Games, #1)' -- Suzanne Collins
Орташа рейтинг: 4.34

Ұсынылған кітаптар:
  1. The Hunger Games Trilogy Boxset (The Hunger G | Suzanne Collins           | sim=0.977
  2. Mockingjay (The Hunger Games, #3)             | Suzanne Collins           | sim=0.950
  3. Catching Fire (The Hunger Games, #2)          | Suzanne Collins           | sim=0.882
  4. The World of the Hunger Games (Hunger Games T | Kate Egan                 | sim=0.773
  5. The Hunger Games Tribute Guide                | Emily Seife               | sim=0.683
Кітап: 'The Hobbit' -- J.R.R. Tolkien
Орташа рейтинг: 4.25

Ұсынылған кітаптар:
  1. J.R.R. Tolkien 4-Book Boxed Set: The Hobbit a | J.R.R. Tolkien            | sim=0.707
  2. The History of the Hobbit, Part One: Mr. Bagg | John D. Rateliff, J.R.R.  | sim=0.657
  3. The Hobbit: Graphic Novel                     | Chuck Dixon, J.R.R. Tolki | sim=0.556
  4. The Children of Húrin                         | J.R.R. Tolkien, Christoph | s

In [15]:
import numpy as np

A = np.array([0.5, 0.8, 0.2, 0.9])
B = np.array([0.4, 0.75, 0.3, 0.85])

# TODO: скалярлық көбейтіндіні есептеңіз
dot_product = np.dot(A, B)

# TODO: A және B нормаларын есептеңіз
norm_a = np.sqrt(np.sum(A**2))
norm_b = np.sqrt(np.sum(B**2))

# TODO: cosine similarity есептеңіз
cos_sim = dot_product / (norm_a * norm_b)

print(f"A . B = {dot_product:.4f}")
print(f"||A|| = {norm_a:.4f}")
print(f"||B|| = {norm_b:.4f}")
print(f"Cosine Similarity = {cos_sim:.6f}")

A . B = 1.6250
||A|| = 1.3191
||B|| = 1.2390
Cosine Similarity = 0.994316


In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# TODO: average_rating және ratings_count бағандарын таңдаңыз
num_features = books[['average_rating', 'ratings_count']].fillna(0).values

# TODO: StandardScaler арқылы стандарттаңыз
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num_features)

# TODO: 0-індекстегі кітап үшін cosine_similarity есептеңіз
idx = 0
sim_num = cosine_similarity(num_scaled[idx:idx+1], num_scaled).flatten()

top5 = sim_num.argsort()[::-1][1:6]
print(f"'{books.loc[idx, 'title']}' -- сандық ерекшеліктер бойынша ұқсас:")
for i, bi in enumerate(top5, 1):
    print(f"  {i}. {books.loc[bi, 'title'][:45]:45s} | "
          f"rating={books.loc[bi, 'average_rating']:.2f}, "
          f"count={books.loc[bi, 'ratings_count']}")

'The Hunger Games (The Hunger Games, #1)' -- сандық ерекшеліктер бойынша ұқсас:
  1. My Sister's Keeper                            | rating=4.06, count=863879
  2. 1984                                          | rating=4.14, count=1956832
  3. Slaughterhouse-Five                           | rating=4.06, count=846488
  4. Anna Karenina                                 | rating=4.02, count=297472
  5. The Night Circus                              | rating=4.03, count=429543


10-тапсырма. Қорытынды сұрақ
Жауап:
TF-IDF тәсілі кітаптың мазмұнына (авторы мен атауына) негізделсе, сандық тәсіл тек оның танымалдылығы мен бағасына қарайды. Сондықтан TF-IDF әлдеқайда жақсы жұмыс істейді, өйткені ол мағыналық жағынан ұқсас кітаптарды ұсынады, ал сандық тәсіл тек статистикасы ұқсас, бірақ тақырыбы мүлдем басқа кітаптарды таңдауы мүмкін.